# Vignette 4: ESR1-D538G Mutation and Midkine Secretion Programs

This vignette demonstrates Module 4's region analysis capabilities using the ESR1/MDK story from patient HCC22-088-P4-S2.

**Research Question**: What contextual factors in ESR1-mutant cancer cells drive MDK (midkine) secretion?

**Key Finding to Explain**: MCF7 cells show increased MDK secretion with D538G mutation, but T47D cells do not. We use Module 4 to identify the "permissive context" genes.

**Workflow**:
1. Discover spatial programs in cancer cells
2. Identify D538G-enriched programs
3. Extract MDK contextual genes
4. Validate with bulk RNA-seq (GSE89888) - interaction effect analysis
5. Validate with ChIP-seq (GSE125117) - ER binding mechanism

## Part 1: Setup and Data Loading

In [ ]:
# Standard imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.insert(0, os.path.abspath(".."))

# CITEgeist imports
from model import CitegeistModel
from model import (
    discover_programs_from_layers,
    analyze_program_regions,
    compare_programs_by_region,
    extract_program_context_genes,
)

# Set plotting defaults
sc.set_figure_params(scanpy=True, fontsize=12)
plt.rcParams['figure.figsize'] = (8, 6)

In [ ]:
# USER CONFIGURATION
LICENSE_FILE = "/ihome/crc/install/gurobi/gurobi1102/linux64/lic/gurobi.lic"
DATA_FOLDER = "/ix1/alee/LO_LAB/General/Lab_Data/20250210_CITEGeistPublicData_GEO_Alex/processed_files/"

# Sample with D538G mutation
SAMPLE_NAME = "HCC22-088-P4-S2_1i_rep"
SAMPLE_PATH = os.path.join(DATA_FOLDER, SAMPLE_NAME, "outs")

In [ ]:
# Load spatial data
adata = sq.read.visium(
    SAMPLE_PATH,
    counts_file='filtered_feature_bc_matrix.h5',
    load_images=True,
    gex_only=False  # Include antibody data
)

print(f"Loaded: {adata.shape[0]} spots x {adata.shape[1]} features")

In [ ]:
# Cell type profiles (same as vignette 2)
cell_profiles = {
    "Cancer Cells": {"Major": ["EPCAM-1"], "Minor": ["SDC1-1", "KRT5-1"]},
    "Macrophages": {"Major": ["CD68-1"], "Minor": ["CD14-1"]},
    "CD4 T Cells": {"Major": ["CD3E-1", "CD4-1"]},
    "CD8 T Cells": {"Major": ["CD3E-1", "CD8A-1"]},
    "B Cells": {"Major": ["MS4A1-1", "CD19-1"]},
    "Endothelial Cells": {"Major": ["PECAM1-1"]},
    "Fibroblasts": {"Major": ["ACTA2-1"]},
}

In [ ]:
# Initialize and run CITEgeist (or load pre-computed results)
model = CitegeistModel(
    sample_name=SAMPLE_NAME,
    adata=adata,
    output_folder="output_vignette4"
)

model.load_cell_profile_dict(cell_profiles)
model.split_adata()
model.filter_gex(nonzero_percentage=0.01, mean_expression_threshold=1.1, min_counts=25)
model.copy_gex_to_protein_adata()
model.preprocess_gex()
model.preprocess_antibody()
model.register_gurobi(LICENSE_FILE)

# Load or run deconvolution
model.append_proportions_to_adata(key="finetuned")
model.append_gex_to_adata(pass_number=1)

prop_gex_adata = model.get_adata()
print(prop_gex_adata)

In [ ]:
# Define D538G mutation regions using basal cytokeratin expression
# (Same approach as vignette 2)
paper_keratins = ["KRT5", "KRT6A", "KRT6B", "KRT14", "KRT16", "KRT17"]
available_keratins = [g for g in paper_keratins if g in prop_gex_adata.var_names]

if available_keratins:
    keratin_idx = [prop_gex_adata.var_names.get_loc(g) for g in available_keratins]
    prop_gex_adata.obs["Basal_Cytokeratin"] = prop_gex_adata.layers["Cancer_Cells_genes_pass1"][:, keratin_idx].sum(axis=1)
    prop_gex_adata.obs["D538G_Mutation"] = prop_gex_adata.obs["Basal_Cytokeratin"] > 0
else:
    # Fallback: use spatial clustering or manual annotation
    print("Warning: Keratin genes not found. Using placeholder D538G annotation.")
    prop_gex_adata.obs["D538G_Mutation"] = False

# Show D538G region distribution
print(f"D538G+ spots: {prop_gex_adata.obs['D538G_Mutation'].sum()}")
print(f"D538G- spots: {(~prop_gex_adata.obs['D538G_Mutation']).sum()}")

In [ ]:
# Visualize D538G regions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.spatial(prop_gex_adata, color="Cancer Cells", ax=axes[0], show=False, title="Cancer Cell Proportion")
sc.pl.spatial(prop_gex_adata, color="D538G_Mutation", ax=axes[1], show=False, title="D538G Mutation Status")

plt.tight_layout()
plt.savefig("figures/vignette4_d538g_regions.png", dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Module 4 Program Discovery

Discover spatial transcriptomic programs in cancer cells using deconvolved expression layers from Module 3.

In [ ]:
# Discover programs in cancer cells
# Using the deconvolved Cancer_Cells layer from Module 3

result = discover_programs_from_layers(
    adata=prop_gex_adata,
    cell_type="Cancer_Cells",
    layer_suffix="_genes_pass1",
    K_programs=5,  # Discover 5 programs
    min_proportion=0.1,  # Only spots with >10% cancer cells
    top_genes=50,
)

print(f"Discovered {len(result.programs)} programs in {result.n_spots_used} spots")
print(f"Reconstruction error: {result.reconstruction_error:.4f}")

In [ ]:
# Visualize program activities spatially
H = result.H  # (K_programs, n_spots)

# Add program activities to adata.obs for visualization
for k in range(len(result.programs)):
    prop_gex_adata.obs[f"Program_{k}"] = 0.0
    # Map back to original spots (result only includes high-cancer spots)
    # This requires matching spot indices - simplified version:
    prop_gex_adata.obs.loc[prop_gex_adata.obs["Cancer Cells"] > 0.1, f"Program_{k}"] = H[k, :]

# Plot program activities
program_cols = [f"Program_{k}" for k in range(len(result.programs))]
sc.pl.spatial(prop_gex_adata, color=program_cols, ncols=3, title=[f"Program {k}" for k in range(len(result.programs))])

In [ ]:
# Show top genes per program
for k, prog in enumerate(result.programs):
    print(f"\nProgram {k} (Moran's I: {prog.spatial_moran_i:.3f}, p={prog.spatial_moran_pvalue:.3e}):")
    print(f"  Top genes: {', '.join(prog.top_genes[:10])}")

## Part 3: Region Enrichment Analysis

Identify which programs are enriched in D538G+ vs D538G- regions.

In [ ]:
# Analyze region enrichment
result = analyze_program_regions(
    result=result,
    adata=prop_gex_adata[prop_gex_adata.obs["Cancer Cells"] > 0.1],  # Match spots used in discovery
    region_column="D538G_Mutation",
    min_spots_per_region=20
)

# Summary table
region_summary = []
for prog in result.programs:
    region_summary.append({
        "Program": prog.program_id,
        "D538G+ Activity": prog.region_enrichment.get("True", 0),
        "D538G- Activity": prog.region_enrichment.get("False", 0),
        "Specificity": prog.region_specificity,
        "P-value": prog.region_pvalue,
        "Enriched In": prog.enriched_region,
    })

region_df = pd.DataFrame(region_summary)
print("\nRegion Enrichment Summary:")
display(region_df)

In [ ]:
# Detailed comparison between regions
comparison_df = compare_programs_by_region(
    result=result,
    adata=prop_gex_adata[prop_gex_adata.obs["Cancer Cells"] > 0.1],
    region_column="D538G_Mutation",
    region_a=True,   # D538G+
    region_b=False,  # D538G-
    top_n_genes=50
)

print("\nProgram Comparison (D538G+ vs D538G-):")
display(comparison_df[["program_id", "mean_activity_a", "mean_activity_b", "fold_change", "pvalue", "top_genes"]])

In [ ]:
# Identify D538G-enriched programs (fold change > 1.5 and p < 0.05)
d538g_enriched = comparison_df[(comparison_df["fold_change"] > 1.5) & (comparison_df["pvalue"] < 0.05)]

print(f"\nFound {len(d538g_enriched)} D538G-enriched programs:")
for _, row in d538g_enriched.iterrows():
    print(f"  Program {row['program_id']}: FC={row['fold_change']:.2f}, p={row['pvalue']:.3e}")

## Part 4: MDK Secretion Context Discovery

Find the program with highest MDK loading and extract co-loaded "contextual factors".

In [ ]:
# Check MDK loading in each program
mdk_loadings = []
W = result.W  # (n_genes, K_programs)

if "MDK" in result.gene_names:
    mdk_idx = result.gene_names.index("MDK")
    for k in range(len(result.programs)):
        mdk_loadings.append({
            "Program": k,
            "MDK_Loading": W[mdk_idx, k],
            "D538G_Enriched": k in d538g_enriched["program_id"].values if len(d538g_enriched) > 0 else False
        })
    
    mdk_df = pd.DataFrame(mdk_loadings).sort_values("MDK_Loading", ascending=False)
    print("MDK Loading by Program:")
    display(mdk_df)
else:
    print("Warning: MDK not found in gene list")
    mdk_df = pd.DataFrame()

In [ ]:
# Identify the "secretion program" - highest MDK loading among D538G-enriched
if len(mdk_df) > 0 and len(d538g_enriched) > 0:
    # Filter to D538G-enriched programs
    d538g_mdk = mdk_df[mdk_df["D538G_Enriched"]]
    
    if len(d538g_mdk) > 0:
        secretion_program_id = d538g_mdk.iloc[0]["Program"]
        print(f"\nIdentified Secretion Program: Program {secretion_program_id}")
        print(f"  MDK Loading: {d538g_mdk.iloc[0]['MDK_Loading']:.4f}")
    else:
        # Fallback: use highest MDK program regardless of D538G enrichment
        secretion_program_id = mdk_df.iloc[0]["Program"]
        print(f"\nNo D538G-enriched programs found. Using highest MDK program: {secretion_program_id}")
else:
    secretion_program_id = 0
    print("Using Program 0 as default")

In [ ]:
# Extract contextual factors - genes co-loaded with MDK
context_genes = extract_program_context_genes(
    result=result,
    program_id=int(secretion_program_id),
    target_gene="MDK",
    top_n=50,
    exclude_target=True
)

print(f"\nTop 50 Contextual Factors (co-loaded with MDK):")
context_df = pd.DataFrame(context_genes, columns=["Gene", "Loading"])
display(context_df.head(20))

In [ ]:
# Check for autocrine receptor genes (SDC4, NCL) in programs
receptor_genes = ["SDC4", "NCL", "PTPRZ1", "LRP1"]  # Known MDK receptors

receptor_loadings = []
for gene in receptor_genes:
    if gene in result.gene_names:
        gene_idx = result.gene_names.index(gene)
        for k in range(len(result.programs)):
            receptor_loadings.append({
                "Gene": gene,
                "Program": k,
                "Loading": W[gene_idx, k]
            })

if receptor_loadings:
    receptor_df = pd.DataFrame(receptor_loadings)
    receptor_pivot = receptor_df.pivot(index="Gene", columns="Program", values="Loading")
    print("\nMDK Receptor Gene Loadings by Program:")
    display(receptor_pivot)

In [ ]:
# Save contextual genes for validation
os.makedirs("output_vignette4", exist_ok=True)
context_df.to_csv("output_vignette4/mdk_contextual_genes.csv", index=False)

# Also save as a simple gene list
with open("output_vignette4/contextual_gene_list.txt", "w") as f:
    for gene, _ in context_genes:
        f.write(f"{gene}\n")

print(f"Saved {len(context_genes)} contextual genes for validation")

## Part 5: Load Bulk RNA-seq Data (GSE89888)

Load RNA-seq data from MCF7 and T47D cell lines with ESR1 WT and D538G mutations.

**Dataset**: GSE89888 - "RNA-seq analysis of ESR1 mutations in T47D and MCF7 cell lines"
- Cell lines: MCF7 (responder - shows MDK upregulation), T47D (non-responder)
- Conditions: WT, D538G mutant
- Treatment: Vehicle, E2
- 4 replicates per condition

In [ ]:
# Configuration for bulk RNA-seq data
# Users should download from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE89888

RNASEQ_DATA_PATH = "data/GSE89888/"  # Update this path

# Expected file structure:
# data/GSE89888/
#   ├── counts_matrix.csv (or .txt)
#   └── sample_metadata.csv

In [ ]:
# Load RNA-seq counts and metadata
# This cell will need adjustment based on actual file format from GEO

try:
    # Try loading processed counts
    rnaseq_counts = pd.read_csv(os.path.join(RNASEQ_DATA_PATH, "counts_matrix.csv"), index_col=0)
    rnaseq_meta = pd.read_csv(os.path.join(RNASEQ_DATA_PATH, "sample_metadata.csv"), index_col=0)
    
    print(f"Loaded RNA-seq: {rnaseq_counts.shape[0]} genes x {rnaseq_counts.shape[1]} samples")
    print(f"\nSample groups:")
    print(rnaseq_meta.groupby(["cell_line", "esr1_status"]).size())
    
except FileNotFoundError:
    print("RNA-seq data not found. Please download from GEO (GSE89888) and update RNASEQ_DATA_PATH.")
    print("\nExpected format:")
    print("  - counts_matrix.csv: genes (rows) x samples (columns)")
    print("  - sample_metadata.csv: sample_id, cell_line (MCF7/T47D), esr1_status (WT/D538G), treatment (veh/E2)")
    
    # Create placeholder for demonstration
    rnaseq_counts = None
    rnaseq_meta = None

In [ ]:
# Filter to D538G vs WT comparison (focus on our mutation of interest)
if rnaseq_meta is not None:
    # Filter to vehicle-treated samples (or E2-treated, depending on question)
    mask = rnaseq_meta["treatment"] == "veh"  # Vehicle treatment
    rnaseq_meta_filtered = rnaseq_meta[mask]
    rnaseq_counts_filtered = rnaseq_counts[rnaseq_meta_filtered.index]
    
    print(f"Filtered to vehicle-treated: {len(rnaseq_meta_filtered)} samples")
    print(rnaseq_meta_filtered.groupby(["cell_line", "esr1_status"]).size())

In [ ]:
# Verify MDK upregulation pattern
if rnaseq_counts is not None and "MDK" in rnaseq_counts.index:
    mdk_expr = rnaseq_counts_filtered.loc["MDK"]
    
    # Merge with metadata
    mdk_data = pd.DataFrame({
        "MDK_expr": mdk_expr,
        "cell_line": rnaseq_meta_filtered["cell_line"],
        "esr1_status": rnaseq_meta_filtered["esr1_status"]
    })
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(data=mdk_data, x="cell_line", y="MDK_expr", hue="esr1_status", ax=ax)
    ax.set_title("MDK Expression: D538G vs WT")
    ax.set_ylabel("MDK Expression (counts)")
    plt.savefig("figures/vignette4_mdk_expression.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # Statistical test
    from scipy.stats import mannwhitneyu
    
    for cell_line in ["MCF7", "T47D"]:
        wt = mdk_data[(mdk_data["cell_line"] == cell_line) & (mdk_data["esr1_status"] == "WT")]["MDK_expr"]
        mut = mdk_data[(mdk_data["cell_line"] == cell_line) & (mdk_data["esr1_status"] == "D538G")]["MDK_expr"]
        
        if len(wt) > 0 and len(mut) > 0:
            stat, pval = mannwhitneyu(mut, wt, alternative="greater")
            fc = mut.mean() / wt.mean() if wt.mean() > 0 else float('inf')
            print(f"{cell_line}: D538G/WT FC = {fc:.2f}, p = {pval:.4f}")

## Part 6: RNA-seq Interaction Validation

Test contextual factor genes for interaction effect: genes that change with D538G mutation **only in MCF7** (responder) but not T47D (non-responder).

In [ ]:
# Interaction effect analysis
# For each contextual gene, test if it shows cell_line x esr1_status interaction

from scipy.stats import mannwhitneyu

def compute_interaction_score(gene, counts_df, meta_df):
    """Compute interaction score for a gene.
    
    Interaction = (D538G - WT in MCF7) - (D538G - WT in T47D)
    Positive = gene responds to D538G more in MCF7 than T47D
    """
    if gene not in counts_df.index:
        return None
    
    expr = counts_df.loc[gene]
    
    results = {}
    for cell_line in ["MCF7", "T47D"]:
        mask_wt = (meta_df["cell_line"] == cell_line) & (meta_df["esr1_status"] == "WT")
        mask_mut = (meta_df["cell_line"] == cell_line) & (meta_df["esr1_status"] == "D538G")
        
        wt_vals = expr[meta_df[mask_wt].index]
        mut_vals = expr[meta_df[mask_mut].index]
        
        results[f"{cell_line}_wt_mean"] = wt_vals.mean()
        results[f"{cell_line}_mut_mean"] = mut_vals.mean()
        results[f"{cell_line}_fc"] = mut_vals.mean() / wt_vals.mean() if wt_vals.mean() > 0 else 1.0
        results[f"{cell_line}_diff"] = mut_vals.mean() - wt_vals.mean()
    
    # Interaction score
    results["interaction"] = results["MCF7_diff"] - results["T47D_diff"]
    
    return results

if rnaseq_counts is not None:
    # Analyze contextual genes
    interaction_results = []
    
    for gene, loading in context_genes:
        result = compute_interaction_score(gene, rnaseq_counts_filtered, rnaseq_meta_filtered)
        if result:
            result["gene"] = gene
            result["spatial_loading"] = loading
            interaction_results.append(result)
    
    interaction_df = pd.DataFrame(interaction_results)
    interaction_df = interaction_df.sort_values("interaction", ascending=False)
    
    print(f"\nAnalyzed {len(interaction_df)} contextual genes for interaction effect:")
    display(interaction_df[["gene", "spatial_loading", "MCF7_fc", "T47D_fc", "interaction"]].head(20))

In [ ]:
# Identify validated contextual genes
# Criteria: positive interaction (more change in MCF7) and MCF7 FC > 1.2

if 'interaction_df' in dir():
    validated_genes = interaction_df[
        (interaction_df["interaction"] > 0) & 
        (interaction_df["MCF7_fc"] > 1.2)
    ]
    
    print(f"\nValidated Contextual Genes ({len(validated_genes)} genes):")
    print("These genes are co-loaded with MDK in spatial data AND show MCF7-specific response to D538G")
    display(validated_genes[["gene", "spatial_loading", "MCF7_fc", "T47D_fc", "interaction"]])

In [ ]:
# Compare to random genes - are contextual genes enriched for interaction effects?
if rnaseq_counts is not None:
    # Sample random genes
    np.random.seed(42)
    random_genes = np.random.choice(
        [g for g in rnaseq_counts.index if g not in [x[0] for x in context_genes]],
        size=min(500, len(rnaseq_counts) - len(context_genes)),
        replace=False
    )
    
    random_results = []
    for gene in random_genes:
        result = compute_interaction_score(gene, rnaseq_counts_filtered, rnaseq_meta_filtered)
        if result:
            random_results.append(result)
    
    random_df = pd.DataFrame(random_results)
    
    # Compare distributions
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(random_df["interaction"], bins=50, alpha=0.5, label=f"Random genes (n={len(random_df)})")
    ax.hist(interaction_df["interaction"], bins=20, alpha=0.7, label=f"Contextual genes (n={len(interaction_df)})")
    ax.axvline(x=0, color='k', linestyle='--')
    ax.set_xlabel("Interaction Score (MCF7 effect - T47D effect)")
    ax.set_ylabel("Count")
    ax.set_title("Contextual Genes Show Stronger MCF7-Specific Response")
    ax.legend()
    plt.savefig("figures/vignette4_interaction_enrichment.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # Statistical test
    from scipy.stats import mannwhitneyu
    stat, pval = mannwhitneyu(interaction_df["interaction"], random_df["interaction"], alternative="greater")
    print(f"\nContextual genes vs random: Mann-Whitney U p = {pval:.4e}")

## Part 7: Load ChIP-seq Data (GSE125117)

Load ER ChIP-seq data to validate whether contextual genes are direct ER targets.

**Dataset**: GSE125117 - "ChIP-seq analysis of genome-edited MCF7 and T47D ESR1 mutant cell models"
- ER binding patterns in WT vs D538G mutant cells
- With and without E2 stimulation

In [ ]:
# Configuration for ChIP-seq data
# Users should download from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE125117

CHIPSEQ_DATA_PATH = "data/GSE125117/"  # Update this path

# Expected file structure:
# data/GSE125117/
#   ├── peaks/  (MACS2 peak files)
#   │   ├── MCF7_WT_E2_peaks.narrowPeak
#   │   ├── MCF7_D538G_E2_peaks.narrowPeak
#   │   └── ...
#   └── gene_er_binding.csv (processed: gene -> ER binding score per condition)

In [ ]:
# Load processed ChIP-seq binding data
# This assumes peaks have been annotated to nearest genes

try:
    er_binding = pd.read_csv(os.path.join(CHIPSEQ_DATA_PATH, "gene_er_binding.csv"), index_col=0)
    
    print(f"Loaded ER binding data: {er_binding.shape[0]} genes")
    print(f"\nConditions: {er_binding.columns.tolist()}")
    
except FileNotFoundError:
    print("ChIP-seq data not found. Please download from GEO (GSE125117) and process.")
    print("\nExpected format: gene_er_binding.csv with columns for each condition")
    print("  - Rows: gene symbols")
    print("  - Columns: MCF7_WT_veh, MCF7_WT_E2, MCF7_D538G_veh, MCF7_D538G_E2, etc.")
    print("  - Values: ER binding score (e.g., peak signal, binary presence)")
    
    er_binding = None

## Part 8: ChIP-seq Mechanistic Validation

Check if validated contextual genes have differential ER binding in D538G vs WT.

In [ ]:
# Analyze ER binding at contextual genes
if er_binding is not None and 'validated_genes' in dir():
    # Check ER binding for validated genes
    er_at_validated = []
    
    for gene in validated_genes["gene"].values:
        if gene in er_binding.index:
            binding = er_binding.loc[gene]
            er_at_validated.append({
                "gene": gene,
                **binding.to_dict()
            })
    
    er_validated_df = pd.DataFrame(er_at_validated)
    
    print(f"\nER Binding at Validated Contextual Genes ({len(er_validated_df)} genes):")
    display(er_validated_df)

In [ ]:
# Identify genes with differential ER binding (D538G vs WT)
if er_binding is not None and len(er_validated_df) > 0:
    # Assuming columns like: MCF7_WT_E2, MCF7_D538G_E2
    # Calculate differential binding
    
    if "MCF7_D538G_E2" in er_validated_df.columns and "MCF7_WT_E2" in er_validated_df.columns:
        er_validated_df["differential_binding"] = (
            er_validated_df["MCF7_D538G_E2"] - er_validated_df["MCF7_WT_E2"]
        )
        
        # Genes with increased ER binding in D538G
        increased_binding = er_validated_df[er_validated_df["differential_binding"] > 0]
        
        print(f"\nGenes with increased ER binding in D538G: {len(increased_binding)}")
        print("These are high-confidence mechanistic candidates:")
        display(increased_binding[["gene", "MCF7_WT_E2", "MCF7_D538G_E2", "differential_binding"]])

## Part 9: Integrated Findings and Conclusions

In [ ]:
# Compile high-confidence permissive factors
# Criteria:
#   1. Co-loaded with MDK in D538G+ spatial program
#   2. Shows interaction effect in RNA-seq (MCF7-specific)
#   3. (Optional) Has differential ER binding

print("="*60)
print("HIGH-CONFIDENCE PERMISSIVE FACTORS FOR MDK SECRETION")
print("="*60)

print(f"\n1. Spatial Discovery:")
print(f"   - Analyzed {len(result.programs)} programs in cancer cells")
print(f"   - Identified {len(d538g_enriched) if 'd538g_enriched' in dir() else 0} D538G-enriched programs")
print(f"   - Extracted {len(context_genes)} genes co-loaded with MDK")

if 'validated_genes' in dir():
    print(f"\n2. RNA-seq Validation:")
    print(f"   - {len(validated_genes)} genes show MCF7-specific D538G response")
    print(f"   - Top validated genes: {', '.join(validated_genes['gene'].head(5).tolist())}")

if 'increased_binding' in dir() and len(increased_binding) > 0:
    print(f"\n3. ChIP-seq Mechanistic Support:")
    print(f"   - {len(increased_binding)} genes have increased ER binding in D538G")
    print(f"   - Direct ER targets: {', '.join(increased_binding['gene'].tolist())}")

In [ ]:
# Create summary figure for paper
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel A: Spatial map of D538G regions
ax = axes[0, 0]
# (Would use sc.pl.spatial here with prop_gex_adata)
ax.set_title("A. D538G Mutation Regions")
ax.text(0.5, 0.5, "[Spatial Plot]", ha='center', va='center', transform=ax.transAxes)

# Panel B: Program region enrichment
ax = axes[0, 1]
if 'comparison_df' in dir():
    ax.bar(range(len(comparison_df)), comparison_df["fold_change"])
    ax.axhline(y=1, color='k', linestyle='--')
    ax.set_xlabel("Program")
    ax.set_ylabel("Fold Change (D538G+ / D538G-)")
ax.set_title("B. Program Enrichment in D538G Regions")

# Panel C: Contextual gene loadings
ax = axes[1, 0]
if len(context_df) > 0:
    ax.barh(range(min(15, len(context_df))), context_df["Loading"].head(15))
    ax.set_yticks(range(min(15, len(context_df))))
    ax.set_yticklabels(context_df["Gene"].head(15))
    ax.invert_yaxis()
    ax.set_xlabel("NMF Loading")
ax.set_title("C. MDK Contextual Genes (Spatial)")

# Panel D: Interaction effect validation
ax = axes[1, 1]
if 'interaction_df' in dir():
    ax.scatter(interaction_df["MCF7_fc"], interaction_df["T47D_fc"], alpha=0.6)
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
    ax.axvline(x=1, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel("D538G/WT Fold Change (MCF7)")
    ax.set_ylabel("D538G/WT Fold Change (T47D)")
    # Highlight validated genes
    if 'validated_genes' in dir():
        ax.scatter(
            validated_genes["MCF7_fc"], 
            validated_genes["T47D_fc"], 
            color='red', s=50, label='Validated'
        )
        ax.legend()
ax.set_title("D. RNA-seq Validation (Interaction)")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/vignette4_summary.png", dpi=300, bbox_inches='tight')
plt.savefig("figures/vignette4_summary.svg", bbox_inches='tight')
plt.show()

print("\nSummary figure saved to figures/vignette4_summary.png")

In [ ]:
# Export all results
output_dir = "output_vignette4"
os.makedirs(output_dir, exist_ok=True)

# Save program comparison
if 'comparison_df' in dir():
    comparison_df.to_csv(f"{output_dir}/program_region_comparison.csv", index=False)

# Save contextual genes
context_df.to_csv(f"{output_dir}/mdk_contextual_genes.csv", index=False)

# Save validated genes
if 'validated_genes' in dir():
    validated_genes.to_csv(f"{output_dir}/validated_contextual_genes.csv", index=False)

# Save interaction analysis
if 'interaction_df' in dir():
    interaction_df.to_csv(f"{output_dir}/interaction_analysis.csv", index=False)

print(f"Results saved to {output_dir}/")

## Conclusions

This vignette demonstrated how Module 4's region analysis capabilities can be used to:

1. **Discover spatial programs** in cell-type-specific expression layers
2. **Identify region-enriched programs** (e.g., D538G-specific programs)
3. **Extract contextual factors** co-expressed with a gene of interest (MDK)
4. **Validate findings** with orthogonal bulk RNA-seq and ChIP-seq data

**Key Findings**:
- MDK is part of a D538G-enriched transcriptional program in cancer cells
- Contextual genes co-loaded with MDK show MCF7-specific response to D538G mutation
- This explains why MCF7 (but not T47D) shows increased MDK secretion with D538G

**Biological Interpretation**:
The permissive factors identified represent the transcriptional context that enables D538G-driven MDK secretion. These genes may be:
- Co-regulated by the mutant ER (direct targets)
- Part of the same signaling network (indirect effects)
- Differentially expressed at baseline between MCF7 and T47D (permissive context)